# TÜV Rheinland control notebook

One GET returns every TÜV location worldwide; we keep type `VIS` with a German `vic_id`. `CONFIRM_LIVE = False` works offline from the newest dev copy in `test_output/`. A live run writes one CSV into `output/<date>_full/`.

In [ ]:
from datetime import datetime
import json

import pandas as pd

from core import geo, paths
from tuev_rheinland import adapter

CONFIRM_LIVE = False  # True = one live GET to tuv.com (1 retry only on network errors / 5xx); False = newest dev raw.json
STARTED = datetime.now()
MODE = "full" if CONFIRM_LIVE else "recon"  # recon exports go to test_output/
pd.set_option("display.width", 200, "display.max_colwidth", 45, "display.max_rows", 60)
print("live" if CONFIRM_LIVE else "offline", "| mode", MODE)

In [ ]:
if CONFIRM_LIVE:
    records, content = adapter.fetch()
    raw_path = adapter.save_raw(content, STARTED)  # dev copy in test_output/<run>_sample/raw.json
else:
    raw_path = adapter.latest_raw()
    records = json.loads(raw_path.read_text(encoding="utf-8"))
print("source:", raw_path.relative_to(paths.project_root()), "|", len(records), "records")

In [ ]:
rec = pd.DataFrame(records)
print("by type:", rec["type"].value_counts().to_dict())
vis = rec[rec["type"] == "VIS"].copy()
parsed = vis["vic_id"].map(adapter.parse_vic)
vis["pool"] = [f"{c}-{p}" if c else "(no vic_id)" for c, p, _ in parsed]
print("\nVIS by external_db x vic pool:")
print(vis.groupby([vis["external_db"].fillna("(null)").replace("", "(empty)"), "pool"]).size().to_string())
print("\nGerman VIS by vic_id country:", int(parsed.map(lambda t: t[0] == "de").sum()), "of", len(vis))
extra = sorted(set().union(*(r.keys() for r in records)) - set(adapter.KNOWN_FIELDS))
print("unknown fields:", extra or "none")
print("unseen types:", sorted(set(rec["type"]) - {"VIS", "GUT", "OFF", "LAB", "HEC", "DLO", "ZUL", "DRI", "TRC", "TEC"}) or "none")

In [ ]:
centroids = geo.load_plz_centroids(paths.geonames_dir() / "DE.txt")
df = adapter.add_distance(adapter.classify(records, STARTED.isoformat(timespec="seconds")), centroids)
scope = df[df["in_scope"]]
print("in scope:", len(scope), scope["site_type"].value_counts().to_dict(), "| needs_review:", int(scope["needs_review"].sum()))

In [ ]:
# landing-page cross-check. Nothing changes automatically; only the rows listed in ADD_FROM_LANDING / FILL_FROM_LANDING (Nikos's decisions).
# Offline = saved copies next to raw.json; live (CONFIRM_LIVE) = one GET per page.
ADD_FROM_LANDING = [("saarland", "66679", "Saarbrücker Straße 221")]  # (region, PLZ, street): second Losheim station, not in the finder JSON
FILL_FROM_LANDING = {"45371": ("saarland", "66333", "Saarwiesenstraße 7")}  # finder record with empty address (title: Völklingen Saarwiesenstraße)
SAARLAND_OK = True
matches = None
landing_dir = raw_path.parent
found, claims, saarland_rows = [], {}, []
for region, url in adapter.LANDING_PAGES.items():
    page = landing_dir / f"landing_{region}.html"
    if CONFIRM_LIVE:
        page.write_bytes(adapter.fetch_page(url))  # one live GET per page, no retry
    if not page.exists():
        print(f"{region}: no saved copy, skipped")
        continue
    rows, claims[region] = adapter.parse_landing(page.read_bytes(), region)
    found.append(adapter.match_landing(rows, records))
    if region == "saarland":
        saarland_rows = rows
if found:
    matches = pd.concat(found, ignore_index=True)
    print(pd.crosstab(matches["region"], matches["status"]).to_string())
    print("page text states:", claims)
    if "saarland" in claims:
        ok, detail = adapter.saarland_check(saarland_rows, claims["saarland"])
        print("Saarland assertion:", "PASS" if ok else "FAIL", "-", detail)
        SAARLAND_OK = ok
    df = adapter.set_saarland(df, matches)
    show = ["region", "standort", "street", "zip", "status", "matched_type", "matched_id", "how", "same_plz_in_json", "title_hint"]
    print("\nNOT matched as VIS (for Nikos to decide):")
    display(matches[matches["status"] != "matched_vis"][show])
    df = adapter.fill_from_landing(df, matches, FILL_FROM_LANDING)
    df = adapter.add_from_landing(df, matches, ADD_FROM_LANDING)
    df = adapter.add_distance(df, centroids)  # fills the state of added rows
    print("address filled from landing page:", FILL_FROM_LANDING, "| added from landing page:", ADD_FROM_LANDING)
    print("operator_group (in scope):", df[df["in_scope"]]["operator_group"].value_counts().to_dict())
    print("matched only by a similar street (please eyeball):")
    display(matches[matches["how"].str.contains("check|PLZ missing")][show[:8]])

In [ ]:
checks = adapter.run_checks(records, df) + [("Saarland landing page: 24 rows and text still says 24", SAARLAND_OK, "")]
PASSED = all(ok for _, ok, _ in checks)
for name, ok, detail in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {detail}")
print("\nRESULT:", "PASS" if PASSED else "FAIL (export will be tagged unverified)")
scope = df[df["in_scope"]]
print("in scope:", len(scope), scope["site_type"].value_counts().to_dict(), "| needs_review:", int(scope["needs_review"].sum()))
review = df[df["needs_review"]]  # includes rows that are NOT exported (in_scope False)
review[["in_scope", "source_id", "site_type", "name", "zip", "plz_distance_km", "needs_review_reason"]]

In [ ]:
csv_path = adapter.export_csv(df, PASSED, MODE, STARTED)
print(csv_path.relative_to(paths.project_root()))

In [ ]:
x = adapter.fsp_crosscheck(df)
print(f"FSP file: {x['file']} | fsp_partner here: {x['here']} | rows in FSP CSV: {x['fsp']} | matched exact: {x['matched']} | matched loose: {x['matched_loose']}")
print(f"only here ({len(x['only_here'])}):", x["only_here"][:10])
print(f"only in FSP ({len(x['only_fsp'])}):", x["only_fsp"][:10])